In [1]:
# ── Imports & paths ────────────────────────────────────────────────────────
import io
import sys
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from linearmodels.panel import PanelOLS
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/Users/jackzipper/QSS20/final_project/final_project_data/'
OUT_DIR  = '/Users/jackzipper/QSS20/final_project/output/'

# Table layout constants
BG     = 'white'
FG     = 'black'
ACCENT = '#444444'
RULE   = '#aaaaaa'
COLS   = [0.04, 0.40, 0.62, 0.82]
Y0     = 0.82
ROW_H  = 0.090


# ── Helper functions ───────────────────────────────────────────────────────

def run_twfe(data, lag_col):
    """Fit a two-way FE PanelOLS (entity + time) with town-clustered SE."""
    df  = data[[lag_col, 'log_aid_spend']].dropna()
    res = PanelOLS(
        dependent      = df['log_aid_spend'],
        exog           = df[[lag_col]],
        entity_effects = True,
        time_effects   = True,
    ).fit(cov_type='clustered', cluster_entity=True)
    return res


def stars(p):
    """Return significance stars: *** p<0.01, ** p<0.05, * p<0.1."""
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''


def table_row(ax, y, cells, bold=False, color=FG, small=False):
    """Render a single row of monospaced text across COLS positions."""
    fs = 9.5 if not small else 8.5
    fw = 'bold' if bold else 'normal'
    aligns = ['left', 'right', 'right', 'right']
    for x, txt, ha in zip(COLS, cells, aligns):
        ax.text(x, y, txt, transform=ax.transAxes,
                color=color, fontsize=fs, fontweight=fw,
                va='top', ha=ha, fontfamily='monospace')


def hline(ax, y, lw=0.8, color=RULE):
    """Draw a horizontal rule spanning the table width."""
    ax.plot([0.03, 0.97], [y, y], color=color, linewidth=lw,
            transform=ax.transAxes, clip_on=False)


def make_regression_table(lags, png_path):
    """Render a 3-model TWFE result table as a PNG.
    lags: list of (result, param_col) tuples in column order.
    """
    fig_t, ax_t = plt.subplots(figsize=(10, 6.2))
    fig_t.patch.set_facecolor(BG)
    ax_t.set_facecolor(BG)
    ax_t.set_axis_off()
    ax_t.set_xlim(0, 1)
    ax_t.set_ylim(0, 1)

    ax_t.text(0.5, 0.98,
              'Effect of Lagged Violence on Aid Spending\n(Town + Month FE, Clustered SE)',
              transform=ax_t.transAxes, color=FG, fontsize=11.5,
              fontweight='bold', ha='center', va='top')

    y = Y0
    hline(ax_t, y, lw=1.2, color='black')
    y -= 0.01
    table_row(ax_t, y, ['', 'Lag 1 month', 'Lag 3 months', 'Lag 6 months'], bold=True, color=ACCENT)
    y -= ROW_H
    hline(ax_t, y, lw=0.6)
    y -= 0.01

    # Coefficient + SE rows
    table_row(ax_t, y, ['Log deaths (lagged)'] + [
        f'{r.params[col]:.4f}{stars(r.pvalues[col])}' for r, col in lags
    ])
    y -= ROW_H * 0.68
    table_row(ax_t, y, [''] + [
        f'({r.std_errors[col]:.4f})' for r, col in lags
    ], color=ACCENT, small=True)
    y -= ROW_H * 0.90

    hline(ax_t, y, lw=0.6)
    y -= 0.01

    # Fit statistics
    table_row(ax_t, y, ['N']           + [str(int(r.nobs))                   for r, _ in lags])
    y -= ROW_H * 0.80
    table_row(ax_t, y, ['R² (within)'] + [f'{r.rsquared_within:.3f}'         for r, _ in lags])
    y -= ROW_H * 0.80
    table_row(ax_t, y, ['Town FE',  'Yes', 'Yes', 'Yes'])
    y -= ROW_H * 0.80
    table_row(ax_t, y, ['Month FE', 'Yes', 'Yes', 'Yes'])
    y -= ROW_H * 0.80

    hline(ax_t, y, lw=1.2, color='black')
    y -= 0.01

    ax_t.text(0.04, y,
              'Clustered standard errors (by town) in parentheses.  *** p<0.01  ** p<0.05  * p<0.1\n'
              'Dependent variable: log(aid spend + 1).  Independent variable: log(deaths + 1), lagged.',
              transform=ax_t.transAxes, color=ACCENT, fontsize=8.0,
              va='top', fontfamily='monospace')

    fig_t.savefig(png_path, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close(fig_t)
    print(f'Saved .png → {png_path}')


def save_txt_summary(lags, txt_path):
    """Write a plain-text regression summary."""
    lines = [
        '=' * 60,
        ' Summary: Effect of log(deaths) on log(aid spend)',
        '=' * 60,
        f"{'Lag':<15} {'β':>8} {'SE':>8} {'p-value':>10} {'Sig':>6} {'R²(within)':>12}",
        '-' * 60,
    ]
    for r, label, col in lags:
        p   = r.pvalues[col]
        sig = stars(p)
        lines.append(
            f'  {label:<13} {r.params[col]:>8.4f} {r.std_errors[col]:>8.4f}'
            f' {p:>10.4f} {sig:>6} {r.rsquared_within:>12.4f}'
        )
    lines += ['', '*** p<0.01  ** p<0.05  * p<0.1',
               'All models include town FE, month FE, and town-clustered standard errors.']
    with open(txt_path, 'w') as f:
        f.write('\n'.join(lines))
    print(f'Saved .txt → {txt_path}')

In [2]:
# ── Load & prep ────────────────────────────────────────────────────────────
# No death-fill: raw log(deaths+1) so zero-death months are informative
panel = pd.read_csv(DATA_DIR + 'violence_aid_merged.csv')
panel['year_month'] = pd.to_datetime(panel['year_month'], format='%Y-%m')
panel = panel.sort_values(['admin2_name', 'year_month'])
print(f'Loaded {len(panel):,} rows | {panel["admin2_name"].nunique()} towns')

panel['log_aid_spend'] = np.log(panel['total_aid_spend'] + 1)
panel['log_deaths']    = np.log(panel['total_deaths']    + 1)

# Build lag columns before setting index
for lag in [1, 3, 6]:
    panel[f'log_deaths_lag{lag}'] = panel.groupby('admin2_name')['log_deaths'].shift(lag)

panel_indexed = panel.set_index(['admin2_name', 'year_month'])
print(f'Lag columns added: log_deaths_lag1, lag3, lag6')

Loaded 10,512 rows | 146 towns
Lag columns added: log_deaths_lag1, lag3, lag6


In [3]:
# ── Fit three lag specifications ───────────────────────────────────────────
res_lag1 = run_twfe(panel_indexed, 'log_deaths_lag1')
res_lag3 = run_twfe(panel_indexed, 'log_deaths_lag3')
res_lag6 = run_twfe(panel_indexed, 'log_deaths_lag6')

lags_named = [
    (res_lag1, 'Lag 1 month',  'log_deaths_lag1'),
    (res_lag3, 'Lag 3 months', 'log_deaths_lag3'),
    (res_lag6, 'Lag 6 months', 'log_deaths_lag6'),
]
lags_table = [(r, col) for r, _, col in lags_named]

for r, label, col in lags_named:
    p = r.pvalues[col]
    print(f'{label}: β={r.params[col]:.4f}, SE={r.std_errors[col]:.4f}, p={p:.4f}{stars(p)}, R²(within)={r.rsquared_within:.3f}')

Lag 1 month: β=-0.3194, SE=0.1294, p=0.0136**, R²(within)=-0.005
Lag 3 months: β=-0.2904, SE=0.1224, p=0.0177**, R²(within)=-0.005
Lag 6 months: β=-0.2781, SE=0.1123, p=0.0133**, R²(within)=-0.004


In [4]:
# ── Export ─────────────────────────────────────────────────────────────────
save_txt_summary(lags_named, OUT_DIR + 'violence_aid_regression.txt')
make_regression_table(lags_table, OUT_DIR + 'violence_aid_regression.png')

Saved .txt → /Users/jackzipper/QSS20/final_project/output/violence_aid_regression.txt
Saved .png → /Users/jackzipper/QSS20/final_project/output/violence_aid_regression.png
